In [1]:
# Change working directory
from pathlib import Path
import os

# Path to file where SWMM model simulation outputs are saved
path = Path(r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Beacon")

os.chdir(path)

print(Path.cwd())

P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Beacon


In [2]:
# Load python libraries
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

In [3]:
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

# Load pre-project report
pre_rpt = read_rpt_file("SSF_SDMP_Beacon_pre_6.8.rpt")

# Extract data
pre = pre_rpt.node_flooding_summary.copy()

# Ensure Node is index
pre.index.name = "Node"

# Rename columns
pre = pre.rename(columns={
    "Hours_Flooded": "Pre Hours Flooded",
    "Maximum_Rate_CFS": "Pre Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Pre Total Flood Vol (MG)",
})

# Keep only needed columns
pre = pre[[
    "Pre Hours Flooded",
    "Pre Max Flood Rate (cfs)",
    "Pre Total Flood Vol (MG)"
]]

pre.head()

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG)
Node,,,
sQ2084,0.19,1.17,0.001
sQ2085,0.22,0.89,0.001
sQ2086,0.44,4.29,0.007
sQ2087,0.44,3.79,0.008
sQ2088,0.43,3.55,0.006


In [4]:
# Load post-project report
post_rpt = read_rpt_file("SSF_SDMP_Beacon_imp1_6.8.rpt")

# Extract data
post = post_rpt.node_flooding_summary.copy()

# Ensure Node is index
post.index.name = "Node"

# Rename columns
post = post.rename(columns={
    "Hours_Flooded": "Post Hours Flooded",
    "Maximum_Rate_CFS": "Post Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Post Total Flood Vol (MG)",
})

# Keep only needed columns
post = post[[
    "Post Hours Flooded",
    "Post Max Flood Rate (cfs)",
    "Post Total Flood Vol (MG)"
]]

post.head()

,Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG)
Node,,,
sQ2084,0.19,1.16,0.001
sQ2085,0.22,0.88,0.001
sQ2086,0.44,4.37,0.007
sQ2087,0.44,3.78,0.008
sQ2088,0.42,3.48,0.006


In [5]:
# Merge on index (Node)
comparison = pre.join(post, how="outer")

# Fill missing values
fill_cols = [
    "Pre Total Flood Vol (MG)",
    "Post Total Flood Vol (MG)",
    "Pre Hours Flooded",
    "Post Hours Flooded"
]

for col in fill_cols:
    comparison[col] = comparison[col].fillna(0)

# -----------------------------
# Reduction calculations
# -----------------------------

# Flood volume reduction
comparison["Total Flood Vol Reduction (MG)"] = (
    comparison["Pre Total Flood Vol (MG)"] - comparison["Post Total Flood Vol (MG)"]
)

comparison["Total Flood Vol Percent Reduction"] = np.where(
    comparison["Pre Total Flood Vol (MG)"] > 0,
    (comparison["Total Flood Vol Reduction (MG)"] / comparison["Pre Total Flood Vol (MG)"]) * 100,
    0
)

# Flood duration reduction
comparison["Hours Flooded Reduction"] = (
    comparison["Pre Hours Flooded"] - comparison["Post Hours Flooded"]
)

comparison["Hours Flooded Percent Reduction"] = np.where(
    comparison["Pre Hours Flooded"] > 0,
    (comparison["Hours Flooded Reduction"] / comparison["Pre Hours Flooded"]) * 100,
    0
)

# Round values
comparison["Total Flood Vol Reduction (MG)"] = comparison["Total Flood Vol Reduction (MG)"].round(3)
comparison["Total Flood Vol Percent Reduction"] = comparison["Total Flood Vol Percent Reduction"].round(1)

comparison["Hours Flooded Reduction"] = comparison["Hours Flooded Reduction"].round(2)
comparison["Hours Flooded Percent Reduction"] = comparison["Hours Flooded Percent Reduction"].round(1)

# Sort by severity
comparison = comparison.sort_values(
    by="Pre Total Flood Vol (MG)",
    ascending=False
)

comparison.head(20)

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG),Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG),Total Flood Vol Reduction (MG),Total Flood Vol Percent Reduction,Hours Flooded Reduction,Hours Flooded Percent Reduction
Node,,,,,,,,,,
sQ2087,0.44,3.79,0.008,0.44,3.78,0.008,0.000,0.0,0.00,0.0
sQ2086,0.44,4.29,0.007,0.44,4.37,0.007,0.000,0.0,0.00,0.0
sQ2093,0.52,3.59,0.006,0.48,2.94,0.005,0.001,16.7,0.04,7.7
sQ2088,0.43,3.55,0.006,0.42,3.48,0.006,0.000,0.0,0.01,2.3
sQ2094,0.35,9.75,0.005,0.34,4.17,0.005,0.000,0.0,0.01,2.9
sQ2095,0.29,3.32,0.003,0.27,2.61,0.003,0.000,0.0,0.02,6.9
sQ2089,0.29,1.81,0.002,0.29,1.81,0.002,0.000,0.0,0.00,0.0
sQ2085,0.22,0.89,0.001,0.22,0.88,0.001,0.000,0.0,0.00,0.0
sQ2084,0.19,1.17,0.001,0.19,1.16,0.001,0.000,0.0,0.00,0.0


In [6]:
# Reset index so Node becomes a column
export_df = comparison.reset_index()

# Export to CSV
export_df.to_csv("flood_comparison.csv", index=False)

print("Exported: flood_comparison.csv")

Exported: flood_comparison.csv


In [7]:
# -----------------------------
# Entire network flood reduction
# -----------------------------

# --- Number of flooded nodes ---
pre_flooded_nodes = (comparison["Pre Hours Flooded"] > 0).sum()
post_flooded_nodes = (comparison["Post Hours Flooded"] > 0).sum()
flooded_nodes_reduction = pre_flooded_nodes - post_flooded_nodes
flooded_nodes_percent_reduction = float(np.where(
    pre_flooded_nodes > 0,
    (flooded_nodes_reduction / pre_flooded_nodes) * 100,
    0
))

# --- Average flood duration (flooded nodes only) ---
pre_avg_duration = comparison.loc[comparison["Pre Hours Flooded"] > 0, "Pre Hours Flooded"].mean()
post_avg_duration = comparison.loc[comparison["Post Hours Flooded"] > 0, "Post Hours Flooded"].mean()
pre_avg_duration = pre_avg_duration if not pd.isna(pre_avg_duration) else 0.0
post_avg_duration = post_avg_duration if not pd.isna(post_avg_duration) else 0.0
avg_duration_reduction = pre_avg_duration - post_avg_duration
avg_duration_percent_reduction = float(np.where(
    pre_avg_duration > 0,
    (avg_duration_reduction / pre_avg_duration) * 100,
    0
))

# --- Volume ---
network_pre_total = comparison["Pre Total Flood Vol (MG)"].sum()
network_post_total = comparison["Post Total Flood Vol (MG)"].sum()
network_reduction_mg = network_pre_total - network_post_total

network_percent_reduction = np.where(
    network_pre_total > 0,
    (network_reduction_mg / network_pre_total) * 100,
    0
)

# -----------------------------
# Print results
# -----------------------------

print("--- Flood Volume ---")
print(f"Pre-Project Network Total Flood Volume: {network_pre_total:.3f} MG")
print(f"Post-Project Network Total Flood Volume: {network_post_total:.3f} MG")
print(f"Network Flood Volume Reduction: {network_reduction_mg:.3f} MG")
print(f"Network Flood Volume Percent Reduction: {network_percent_reduction:.1f}%")

print("\n--- Flooded Nodes ---")
print(f"Pre-Project Number of Flooded Nodes: {pre_flooded_nodes}")
print(f"Post-Project Number of Flooded Nodes: {post_flooded_nodes}")
print(f"Flooded Nodes Reduction: {flooded_nodes_reduction}")
print(f"Flooded Nodes Percent Reduction: {flooded_nodes_percent_reduction:.1f}%")

print("\n--- Average Flood Duration (Flooded Nodes Only) ---")
print(f"Pre-Project Avg Flood Duration: {pre_avg_duration:.2f} hrs")
print(f"Post-Project Avg Flood Duration: {post_avg_duration:.2f} hrs")
print(f"Avg Flood Duration Reduction: {avg_duration_reduction:.2f} hrs")
print(f"Avg Flood Duration Percent Reduction: {avg_duration_percent_reduction:.1f}%")


--- Flood Volume ---
Pre-Project Network Total Flood Volume: 0.040 MG
Post-Project Network Total Flood Volume: 0.038 MG
Network Flood Volume Reduction: 0.002 MG
Network Flood Volume Percent Reduction: 5.0%

--- Flooded Nodes ---
Pre-Project Number of Flooded Nodes: 12
Post-Project Number of Flooded Nodes: 11
Flooded Nodes Reduction: 1
Flooded Nodes Percent Reduction: 8.3%

--- Average Flood Duration (Flooded Nodes Only) ---
Pre-Project Avg Flood Duration: 0.28 hrs
Post-Project Avg Flood Duration: 0.29 hrs
Avg Flood Duration Reduction: -0.01 hrs
Avg Flood Duration Percent Reduction: -5.2%
